In [ ]:
from pathlib import Path
from time import perf_counter

import pandas as pd
import pyomo.environ as pyo


def load_coverage_table(
    matrix_path,
    max_cover_dist_m=10_000,
    target_col="target_id",
    source_col="source_id",
    distance_col="total_dist",
):
    matrix = pd.read_parquet(
        matrix_path,
        columns=[target_col, source_col, distance_col],
    )
    return matrix.loc[
        matrix[distance_col] <= max_cover_dist_m,
        [target_col, source_col, distance_col],
    ].drop_duplicates()


def build_max_covering_model(
    coverage,
    population_path,
    p=10,
    target_col="target_id",
    source_col="source_id",
    population_id_col="ID",
    population_weight_col="population",
):
    population = pd.read_parquet(
        population_path,
        columns=[population_id_col, population_weight_col],
    )
    weights = population.set_index(population_id_col)[population_weight_col].astype(float).to_dict()

    targets = sorted(population[population_id_col].unique())
    sources = sorted(coverage[source_col].unique())
    cover_by_target = coverage.groupby(target_col)[source_col].apply(list).to_dict()

    model = pyo.ConcreteModel()
    model.I = pyo.Set(initialize=targets)
    model.J = pyo.Set(initialize=sources)
    model.x = pyo.Var(model.J, domain=pyo.Binary)
    model.y = pyo.Var(model.I, domain=pyo.Binary)

    model.objective = pyo.Objective(
        expr=sum(weights.get(i, 0.0) * model.y[i] for i in model.I),
        sense=pyo.maximize,
    )

    def coverage_rule(m, i):
        neighborhood = cover_by_target.get(i, [])
        if not neighborhood:
            return m.y[i] <= 0
        return m.y[i] <= sum(m.x[j] for j in neighborhood)

    model.coverage = pyo.Constraint(model.I, rule=coverage_rule)
    model.facility_limit = pyo.Constraint(expr=sum(model.x[j] for j in model.J) == p)

    return model


def solve_and_summarize(model, solver_name, tee=False, time_limit_s=300):
    solver = pyo.SolverFactory(solver_name)
    if solver_name == "gurobi":
        solver.options["TimeLimit"] = time_limit_s
        solver.options["Threads"] = 8
    elif solver_name in {"appsi_highs", "highs"}:
        solver.options["time_limit"] = time_limit_s
        solver.options["threads"] = 8

    start = perf_counter()
    result = solver.solve(model, tee=tee)
    solve_time_s = perf_counter() - start

    selected_sources = [j for j in model.J if pyo.value(model.x[j]) > 0.5]
    covered_targets = [i for i in model.I if pyo.value(model.y[i]) > 0.5]

    return {
        "solver": solver_name,
        "status": str(result.solver.status),
        "termination": str(result.solver.termination_condition),
        "solve_time_s": solve_time_s,
        "selected_sources": selected_sources,
        "covered_targets": covered_targets,
        "covered_population": pyo.value(model.objective),
        "model": model,
    }


In [ ]:
data = Path(r'C:\local\Download_Depot\east-timor_data\outputs')
run_tag = 'pop_1_sample_1_max_none_agg_8_maxdist_10000_amenity_all_candidates_spacing_5000_maxsnap_5000'

matrix_path = data / f'distance_matrix_{run_tag}.parquet'
population_path = data / f'population_{run_tag}.parquet'
sources_path = data / f'sources_{run_tag}.parquet'

max_cover_dist_m = 10_000
p_facilities = 20

coverage = load_coverage_table(
    matrix_path=matrix_path,
    max_cover_dist_m=max_cover_dist_m,
)

solution_summaries = []
solved_models = {}
for solver_name in ['gurobi', 'appsi_highs']:
    model = build_max_covering_model(
        coverage=coverage,
        population_path=population_path,
        p=p_facilities,
    )
    summary = solve_and_summarize(model, solver_name=solver_name, tee=False)
    solved_models[solver_name] = summary['model']
    solution_summaries.append({key: value for key, value in summary.items() if key != 'model'})

solution_summary_df = pd.DataFrame([
    {
        'solver': item['solver'],
        'status': item['status'],
        'termination': item['termination'],
        'solve_time_s': item['solve_time_s'],
        'covered_population': item['covered_population'],
        'opened_facilities': len(item['selected_sources']),
        'covered_targets': len(item['covered_targets']),
    }
    for item in solution_summaries
])
solution_summary_df


In [ ]:
from distance_pipeline.viz import build_solution_layers, plot_max_cover_solution


In [ ]:
base_path_for_document = Path(r'C:\Users\joaqu\Dropbox\Apps\Overleaf\Real Life Distance Generator')
figures_path = base_path_for_document / 'figures'
optimization_figure_path = figures_path / 'timor_leste_max_cover_solution.png'

model = solved_models['gurobi']
solution = build_solution_layers(
    model=model,
    matrix_path=matrix_path,
    matrix=coverage,
    population_path=population_path,
    sources_path=sources_path,
    max_cover_dist_m=max_cover_dist_m,
)

fig, ax = plot_max_cover_solution(
    solution,
    title='Timor-Leste maximum covering solution, 10 km threshold',
    output_path=optimization_figure_path,
)
